# Batch Droplet Analysis

Outputs: **`batch_combined.csv`** (all droplets) and **`batch_summary.csv`** (per-image stats)

**Steps:** Run Cell 1 → pick folder + adjust params in Cell 2 → click **▶ Run Batch** → click **▶ Show Plots** / **💾 Save CSVs**

In [ ]:
# Setup — run once
import importlib, batch_pipeline, droplet_pipeline
importlib.reload(droplet_pipeline)
importlib.reload(batch_pipeline)
from batch_pipeline import find_czi_files, run_batch, DEFAULT_PARAMS
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import os
from droplet_pipeline import HEX_COLORS
print('Ready.')

In [ ]:
B = {}   # shared state: combined, summary, labels

# ── Folder picker ─────────────────────────────────────────────────────────────
dir_input = widgets.Text(
    placeholder='Paste folder path here (overrides picker below)',
    description='',
    layout=widgets.Layout(width='500px'),
)
display(dir_input)

try:
    from ipyfilechooser import FileChooser
    fc_dir = FileChooser(
        path=os.getcwd(),
        title="Select folder to scan for .czi files",
        show_hidden=False, select_default=True, show_only_dirs=True,
    )
    def _get_root():
        txt = dir_input.value.strip().strip("\"'")
        if txt: return txt
        return fc_dir.selected_path or os.getcwd()
    display(fc_dir)
except ImportError:
    print("ipyfilechooser not installed — scanning current directory.")
    def _get_root():
        txt = dir_input.value.strip().strip("\"'")
        return txt if txt else os.getcwd()

# ── Parameter sliders ─────────────────────────────────────────────────────────
_sw = {"description_width": "150px"}
_lw = widgets.Layout(width="420px")
param_sliders = {
    "pixel_size_um":    widgets.FloatSlider(value=0.312, min=0.05,  max=2.0,  step=0.001, description="Pixel size (µm)",   style=_sw, layout=_lw),
    "sigma":            widgets.FloatSlider(value=1.0,   min=0.3,   max=5.0,  step=0.1,   description="Blur sigma",        style=_sw, layout=_lw),
    "otsu_scale":       widgets.FloatSlider(value=1.0,   min=0.3,   max=1.5,  step=0.05,  description="Otsu scale",         style=_sw, layout=_lw),
    "min_size":         widgets.IntSlider  (value=20,    min=5,     max=300,  step=5,     description="Min size (px²)",    style=_sw, layout=_lw),
    "min_dist":         widgets.IntSlider  (value=5,     min=1,     max=50,   step=1,     description="Min dist (px)",     style=_sw, layout=_lw),
    "excl_factor":      widgets.FloatSlider(value=2.0,   min=1.0,   max=6.0,  step=0.1,   description="Excl. factor",      style=_sw, layout=_lw),
    "reliability_sigma":widgets.FloatSlider(value=1.0,   min=0.3,   max=5.0,  step=0.1,   description="Reliability σ",     style=_sw, layout=_lw),
    "bright_factor":    widgets.FloatSlider(value=2.0,   min=1.0,   max=10.0, step=0.1,   description="Bright factor",     style=_sw, layout=_lw),
    "bright_pct":       widgets.FloatSlider(value=98.0,  min=80.0,  max=99.9, step=0.1,   description="Bright pct",        style=_sw, layout=_lw),
    "thresh_offset_sigma": widgets.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.1,    description="Thr offset \u03c3",      style=_sw, layout=_lw),
    "margin_sigma":     widgets.FloatSlider(value=0.0,   min=0.0,   max=1.0,  step=0.01,  description="Margin \u03c3",          style=_sw, layout=_lw),
    "snr_threshold":    widgets.FloatSlider(value=2.0,   min=0.0,   max=5.0,  step=0.1,   description="SNR threshold",     style=_sw, layout=_lw),
}
params_box = widgets.VBox(
    [widgets.HTML("<b>Analysis parameters</b>")] + list(param_sliders.values()),
    layout=widgets.Layout(margin="8px 0")
)

# ── Manual threshold override ─────────────────────────────────────────────────
_ch_labels = ["Ch1 red", "Ch2 yellow", "Ch3 cyan", "Ch4 blue"]
thresh_override_chk = widgets.Checkbox(
    value=False, description="Override channel thresholds",
    indent=False, layout=widgets.Layout(width="260px"),
)
thresh_override_boxes = [
    widgets.FloatText(value=0.0, description=f"{lbl}:",
                      style={"description_width": "90px"},
                      layout=widgets.Layout(width="220px"))
    for lbl in _ch_labels
]
thresh_override_row = widgets.HBox(thresh_override_boxes,
                                   layout=widgets.Layout(margin="0 0 4px 0"))
def _toggle_thresh(change):
    thresh_override_row.layout.display = "" if change["new"] else "none"
thresh_override_chk.observe(_toggle_thresh, names="value")
thresh_override_row.layout.display = "none"
thresh_override_box = widgets.VBox(
    [widgets.HTML("<b>Manual channel thresholds</b> (leave Override unchecked to use auto-GMM)"),
     thresh_override_chk, thresh_override_row],
    layout=widgets.Layout(margin="8px 0", border="1px solid #555", padding="6px"),
)

run_btn = widgets.Button(
    description="▶ Run Batch", button_style="primary",
    layout=widgets.Layout(width="160px", height="36px", margin="8px 0"),
)
run_out = widgets.Output()

def _run(_):
    run_out.clear_output(wait=True)
    with run_out:
        PARAMS = {k: s.value for k, s in param_sliders.items()}
        if thresh_override_chk.value:
            PARAMS["manual_thresholds"] = [b.value for b in thresh_override_boxes]
        root  = _get_root()
        files = find_czi_files(root)
        if not files:
            print(f"No .czi files found in:\n  {root}")
            return
        labels = {f: f"Data {i+1}" for i, f in enumerate(files)}
        B["labels"] = labels
        print(f"Found {len(files)} file(s):")
        for f, lbl in labels.items():
            print(f"  {lbl}: {os.path.basename(f)}")
        print()
        combined, summary, errors = run_batch(files, params=PARAMS)
        fname_to_label = {os.path.basename(f): lbl for f, lbl in labels.items()}
        combined["label"] = combined["filename"].map(fname_to_label)
        summary["label"]  = summary["filename"].map(fname_to_label)
        # parse channel reliability into boolean columns
        for ch in range(1, 5):
            summary[f"ch{ch}_reliable"] = ~summary["unreliable_ch"].str.contains(f"ch{ch}", na=False)
        B["combined"] = combined
        B["summary"]  = summary
        if errors:
            print("Failed:")
            for e in errors:
                print(f"  {e['filename']}: {e['error']}")
        cols = ["label", "filename", "n_droplets", "n_confident", "confident_frac",
                "n_classes", "mean_radius_um", "mean_volume_um3", "bright_red_frac", "unreliable_ch"]
        display(summary[cols])
        n_tot  = combined["label"].count()  # total rows across all images
        n_conf_tot = int((combined["min_confidence"] > 0.05).sum()) if "min_confidence" in combined.columns else n_tot
        frac_tot = f"{n_conf_tot/n_tot*100:.0f}%" if n_tot else "\u2014"
        print(f"\nTotal: {n_tot} droplets across all images")
        print(f"Confident (>0.05): {n_conf_tot} ({frac_tot})")
        print("\nFinished!")

run_btn.on_click(_run)
display(params_box, thresh_override_box, run_btn, run_out)

In [ ]:
plot_btn = widgets.Button(
    description="▶ Show Plots", button_style="info",
    layout=widgets.Layout(width="160px", height="36px"),
)
save_btn = widgets.Button(
    description="💾 Save CSVs", button_style="success",
    layout=widgets.Layout(width="160px", height="36px"),
)
save_plots_btn = widgets.Button(
    description="📊 Save Plots", button_style="success",
    layout=widgets.Layout(width="160px", height="36px"),
)
action_out = widgets.Output()

def _do_plots(save_prefix=None):
    _n = [0]
    def _ms(fig):
        if save_prefix:
            _n[0] += 1
            fig.savefig(f"{save_prefix}_{_n[0]:02d}.png", dpi=150, bbox_inches="tight")
    if "summary" not in B:
        print("Run the batch first.")
        return
    summary_df = B["summary"]
    combined_df = B["combined"]
    frac_cols  = [f"frac_class{c:02d}" for c in range(16)]
    labels     = summary_df["label"].tolist()
    n_images   = len(labels)
    frac_mat   = summary_df[frac_cols].values   # (n_images, 16)

    # ── 0. Confident vs low-confidence droplets per image ─────────────────
    if "min_confidence" in combined_df.columns:
        conf_counts = (combined_df[combined_df["min_confidence"] > 0.05]
                       .groupby("label").size().reindex(labels, fill_value=0))
    else:
        conf_counts = combined_df.groupby("label").size().reindex(labels, fill_value=0)
    total_counts = combined_df.groupby("label").size().reindex(labels, fill_value=0)
    low_counts   = total_counts - conf_counts

    fig, ax = plt.subplots(figsize=(max(5, n_images * 1.0), 4), dpi=150)
    ax.bar(labels, conf_counts, color="#4caf50", label="Confident (>0.05)", edgecolor="#333", linewidth=0.5)
    ax.bar(labels, low_counts,  color="#e57373", label="Low confidence",
           bottom=conf_counts, edgecolor="#333", linewidth=0.5)
    for i, lbl in enumerate(labels):
        total = total_counts[lbl]
        pct   = 100 * conf_counts[lbl] / total if total > 0 else 0
        ax.text(i, total, f"{pct:.0f}%", ha="center", va="bottom", fontsize=8)
    ax.set_ylim(0, total_counts.max() * 1.15)
    ax.set_ylabel("Number of droplets")
    ax.set_title("Confident vs low-confidence droplets per image")
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), borderaxespad=0, fontsize=8, framealpha=0.9)
    plt.tight_layout()
    _ms(fig)
    plt.show()

    # ── 1. Class composition + total volume (side by side) ─────────────────
    vol_pivot = (combined_df.groupby(["label","code_int"])["volume_um3"]
                 .sum().unstack(fill_value=0))
    present_cls = list(range(16))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(max(10, n_images * 1.4), 5), dpi=150)
    for ax, data_fn, ylabel, title in [
        (ax1,
         lambda cls: frac_mat[:, cls],
         "Fraction of droplets", "Class composition per image"),
        (ax2,
         lambda cls: vol_pivot[cls].values.astype(float) if cls in vol_pivot.columns else np.zeros(n_images),
         "Total volume (µm³)", "Total volume by class per image"),
    ]:
        bottom = np.zeros(n_images)
        for cls in present_cls:
            vals = data_fn(cls)
            ax.bar(labels, vals, bottom=bottom, color=HEX_COLORS[cls],
                   label=str(cls), edgecolor="#333", linewidth=0.4)
            bottom += vals
        ax.set_ylabel(ylabel)
        ax.set_title(title, fontsize=9)
        ax.legend(title="class", bbox_to_anchor=(1.01, 1), loc="upper left",
                  fontsize=7, ncol=2)
    plt.tight_layout()
    _ms(fig)
    plt.show()

    # ── 2. Count by class + mean radius by class (side by side) ───────────
    count_pivot = (combined_df.groupby(["label","code_int"]).size()
                   .unstack(fill_value=0))
    radius_pivot = (combined_df.groupby(["label","code_int"])["radius_um"]
                    .mean().unstack(fill_value=np.nan))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(max(10, n_images * 1.4), 5), dpi=150)

    # count stacked bar
    bottom = np.zeros(n_images)
    for cls in present_cls:
        vals = count_pivot[cls].values.astype(float) if cls in count_pivot.columns else np.zeros(n_images)
        ax1.bar(labels, vals, bottom=bottom, color=HEX_COLORS[cls],
                label=str(cls), edgecolor="#333", linewidth=0.4)
        bottom += vals
    ax1.set_ylabel("Number of droplets")
    ax1.set_title("Droplet count by class per image", fontsize=9)
    ax1.legend(title="class", bbox_to_anchor=(1.01, 1), loc="upper left",
               fontsize=7, ncol=2)

    # bubble chart: x=class, y=image, class color, size = mean radius, text label
    def _lum(h): r,g,b=int(h[1:3],16),int(h[3:5],16),int(h[5:7],16); return 0.299*r+0.587*g+0.114*b
    xs, ys, means, cls_list = [], [], [], []
    for ci, cls in enumerate(present_cls):
        for ii, lbl in enumerate(labels):
            grp = combined_df[(combined_df.label==lbl) & (combined_df.code_int==cls)]["radius_um"]
            if len(grp) > 0:
                xs.append(ci); ys.append(ii); means.append(grp.mean()); cls_list.append(cls)
    if means:
        vmax2 = max(means)
        bubble_sizes  = [max(60, (m / vmax2) * 900) for m in means]
        bubble_colors = [HEX_COLORS[c % 16] for c in cls_list]
        ax2.scatter(xs, ys, s=bubble_sizes, c=bubble_colors,
                    edgecolors="#333", linewidth=0.8, alpha=0.9)
        for xi, yi, m, cls in zip(xs, ys, means, cls_list):
            tc = "black" if _lum(HEX_COLORS[cls % 16]) > 128 else "white"
            ax2.text(xi, yi, f"{m:.2f}", ha="center", va="center",
                     fontsize=6, color=tc, fontweight="bold")
    ax2.set_xticks(range(len(present_cls)))
    ax2.set_xticklabels([str(c) for c in present_cls], fontsize=8)
    ax2.set_yticks(range(n_images))
    ax2.set_yticklabels(labels, fontsize=8)
    ax2.set_xlim(-0.5, len(present_cls) - 0.5)
    ax2.set_ylim(-0.5, n_images - 0.5)
    ax2.set_xlabel("Class")
    ax2.set_ylabel("Image")
    ax2.set_title("Mean radius (µm) by class × image", fontsize=9)
    ax2.grid(True, linestyle="--", alpha=0.3)
    plt.tight_layout()
    _ms(fig)
    plt.show()

    # ── 3. Combined (pooled) plots ─────────────────────────────────────────
    counts_all = combined_df.groupby("code_int").size().reindex(range(16), fill_value=0)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

    # 3a. Pooled class composition bar
    ax = axes[0]
    ax.bar(counts_all.index, counts_all.values,
           color=[HEX_COLORS[i] for i in range(16)],
           edgecolor="#555", linewidth=0.6)
    ax.set_xlabel("Class")
    ax.set_ylabel("Total droplets (all images)")
    ax.set_title("Pooled class composition")
    ax.set_xticks(range(16))

    # 3b. Radius distribution by class (boxplot)
    ax = axes[1]
    cls_data = [combined_df.loc[combined_df.code_int == c, "radius_um"].values
                for c in present_cls]
    bp = ax.boxplot(cls_data, positions=present_cls, widths=0.6,
                    patch_artist=True,
                    medianprops=dict(color="white", linewidth=1.5),
                    whiskerprops=dict(color="#888"),
                    capprops=dict(color="#888"),
                    flierprops=dict(marker=".", color="#888", markersize=3))
    for patch, cls in zip(bp["boxes"], present_cls):
        patch.set_facecolor(HEX_COLORS[cls])
    ax.set_xlabel("Class")
    ax.set_ylabel("Radius (µm)")
    ax.set_title("Pooled radius distribution by class")
    ax.set_xticks(present_cls)
    plt.tight_layout()
    _ms(fig)
    plt.show()

def _plot(_):
    action_out.clear_output(wait=True)
    with action_out:
        _do_plots()

def _save_plots(_):
    action_out.clear_output(wait=True)
    with action_out:
        if "summary" not in B:
            print("Run the batch first."); return
        root = _get_root()
        import os as _os
        prefix = _os.path.join(root, "batch_plots")
        _do_plots(save_prefix=prefix)
        import glob as _gl
        saved = sorted(_gl.glob(prefix + "_??.png"))
        print(f"Saved {len(saved)} plot(s) → {_os.path.basename(prefix)}_*.png")

def _save(_):
    action_out.clear_output(wait=True)
    with action_out:
        if "combined" not in B:
            print("Run the batch first.")
            return
        import os as _os
        root = _get_root()
        B["combined"].to_csv(_os.path.join(root, "batch_combined.csv"), index=False)
        B["summary"].to_csv(_os.path.join(root, "batch_summary.csv"),  index=False)
        print(f"Saved batch_combined.csv  ({len(B['combined'])} rows) → {root}")
        print(f"Saved batch_summary.csv   ({len(B['summary'])} rows) → {root}")
        if "labels" in B:
            print("\nLabel mapping:")
            for f, lbl in B["labels"].items():
                print(f"  {lbl} = {os.path.basename(f)}")

plot_btn.on_click(_plot)
save_btn.on_click(_save)
save_plots_btn.on_click(_save_plots)
display(widgets.HBox([plot_btn, save_btn, save_plots_btn]), action_out)